# Módulo 11.1 — Autoencoders
**Sergio Gevatschnaider** · laboratorio sintético, reproducible.

Objetivo: comparar PCA con un autoencoder neuronal 2→1→2 entrenado realmente, evaluar reconstrucción fuera de muestra y explicar el error de anomalías. Los datos no corresponden a empresas reales.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch import nn
np.random.seed(21); torch.manual_seed(21)
n=650
x=np.random.normal(0,1,n)
X=np.c_[x,0.7*x+0.28*np.sin(3*x)+np.random.normal(0,.16,n)]
train_idx, other_idx=train_test_split(np.arange(n),test_size=.40,random_state=21)
val_idx, test_idx=train_test_split(other_idx,test_size=.50,random_state=21)
scaler=StandardScaler().fit(X[train_idx])
Xtr=scaler.transform(X[train_idx]).astype('float32')
Xva=scaler.transform(X[val_idx]).astype('float32')
Xte=scaler.transform(X[test_idx]).astype('float32')
print('train',Xtr.shape,'val',Xva.shape,'test',Xte.shape)

## Línea base: PCA
Se ajusta exclusivamente en el conjunto de entrenamiento. No comparar pérdidas in-sample contra test sin advertencia.

In [ ]:
pca=PCA(n_components=1).fit(Xtr)
Xhat_pca=pca.inverse_transform(pca.transform(Xte))
mse_pca=np.mean((Xte-Xhat_pca)**2)
print('PCA MSE test:',round(mse_pca,5))

## Autoencoder no lineal: ajuste, selección mediante validación y evaluación final
El conjunto de **test queda reservado**. El número de épocas se decide con validación y paciencia (early stopping); el test se usa una sola vez para comunicar resultados.


In [ ]:
import copy
model=nn.Sequential(nn.Linear(2,6),nn.Tanh(),nn.Linear(6,1),nn.Tanh(),nn.Linear(1,6),nn.Tanh(),nn.Linear(6,2))
opt=torch.optim.Adam(model.parameters(),lr=.01,weight_decay=1e-4)
tr=torch.tensor(Xtr); va=torch.tensor(Xva); te=torch.tensor(Xte)
history=[]; best=float('inf'); best_state=None; wait=0; patience=35
for epoch in range(500):
    model.train(); opt.zero_grad()
    loss=((model(tr)-tr)**2).mean();loss.backward();opt.step()
    model.eval()
    with torch.no_grad(): val_loss=((model(va)-va)**2).mean().item()
    history.append((loss.item(),val_loss))
    if val_loss<best-1e-6:
        best=val_loss; best_state=copy.deepcopy(model.state_dict());wait=0
    else: wait+=1
    if wait>=patience: break
model.load_state_dict(best_state)
with torch.no_grad(): recon=model(te).numpy()
mse_ae=np.mean((Xte-recon)**2)
print('Épocas:',len(history),'mejor MSE validación:',round(best,5))
print('PCA MSE test:',round(mse_pca,5),'AE MSE test:',round(mse_ae,5))
plt.plot(history);plt.legend(['train','validación']);plt.title('Curva de aprendizaje (MSE)');plt.xlabel('Época');plt.show()


## Reconstrucción y anomalías
El error de reconstrucción no demuestra fraude: define un puntaje que requiere calibración, supervisión y análisis de falsos positivos.

In [ ]:
fig,ax=plt.subplots(figsize=(7,5));ax.scatter(Xte[:,0],Xte[:,1],s=10,label='original');ax.scatter(recon[:,0],recon[:,1],s=10,alpha=.5,label='reconstrucción');ax.legend();plt.show()
score=np.mean((Xte-recon)**2,axis=1)
with torch.no_grad(): recon_val=model(va).numpy()
cut=np.quantile(np.mean((Xva-recon_val)**2,axis=1),.95)
print('Umbral exploratorio sobre validación:',round(cut,5),'alertas en test:',int((score>cut).sum()))

## Reflexión
1. ¿Es la mejora del AE sobre PCA robusta a semillas y arquitectura?
2. ¿Qué resulta del uso de datos no vistos?
3. ¿Cómo calibrarías el costo de alertas?
4. ¿En qué casos el código latente no es interpretable?

## Fundamento matemático: derivada verificable
Para el caso escalar: \(z=w_e x,\;\hat x=w_d z,\; L=\frac12(x-\hat x)^2\). La regla de la cadena proporciona
\(\partial L/\partial w_e=(\hat x-x)w_d x\) y \(\partial L/\partial w_d=(\hat x-x)z\).
Vamos a contrastar el gradiente analítico con diferencias finitas (verificación local; no reemplaza autograd).

In [ ]:
x=2.;we=.5;wd=.8
def toy_loss(we,wd):return .5*(x-wd*we*x)**2
z=we*x;hat=wd*z
analytic_we=(hat-x)*wd*x
analytic_wd=(hat-x)*z
eps=1e-6
numeric_we=(toy_loss(we+eps,wd)-toy_loss(we-eps,wd))/(2*eps)
numeric_wd=(toy_loss(we,wd+eps)-toy_loss(we,wd-eps))/(2*eps)
print('Encoder:',analytic_we,numeric_we,'Decoder:',analytic_wd,numeric_wd)
assert np.allclose([analytic_we,analytic_wd],[numeric_we,numeric_wd],atol=1e-7)

## Experimento geométrico: la curvatura como caso difícil para PCA
Usamos una curva de dimensión intrínseca 1 con ruido. PCA y el AE se entrenan con exactamente el mismo train y escalador. Se evalúa el **mismo test**. No se debe concluir de antemano que AE ganará: el resultado depende de hiperparámetros y optimización.

In [ ]:
rng=np.random.default_rng(52)
t=rng.uniform(-1.2,1.2,size=900)
arc=np.c_[2*np.sin(t),np.cos(t)]+rng.normal(0,.05,(900,2))
it,iret=train_test_split(np.arange(len(arc)),test_size=.4,random_state=52)
iv,ie=train_test_split(iret,test_size=.5,random_state=52)
sc=StandardScaler().fit(arc[it]);ct=torch.tensor(sc.transform(arc[it]),dtype=torch.float32)
cv=torch.tensor(sc.transform(arc[iv]),dtype=torch.float32)
cx=torch.tensor(sc.transform(arc[ie]),dtype=torch.float32)
pc=PCA(n_components=1).fit(ct.numpy())
p_error=np.mean((cx.numpy()-pc.inverse_transform(pc.transform(cx.numpy())))**2)
net=nn.Sequential(nn.Linear(2,20),nn.Tanh(),nn.Linear(20,1),nn.Tanh(),nn.Linear(1,20),nn.Tanh(),nn.Linear(20,2))
adam=torch.optim.Adam(net.parameters(),lr=.012,weight_decay=1e-5)
best=1e9;state=None;wait=0
for epoch in range(700):
    net.train();adam.zero_grad();l=((net(ct)-ct)**2).mean();l.backward();adam.step()
    net.eval()
    with torch.no_grad():v=((net(cv)-cv)**2).mean().item()
    if v<best-1e-6:best=v;state=copy.deepcopy(net.state_dict());wait=0
    else:wait+=1
    if wait>=70:break
net.load_state_dict(state)
with torch.no_grad():prediction=net(cx).numpy()
a_error=np.mean((cx.numpy()-prediction)**2)
print('PCA test:',round(p_error,5),'AE test:',round(a_error,5),'épocas:',epoch+1)
fig,ax=plt.subplots(figsize=(6,5))
ax.scatter(cx[:,0],cx[:,1],s=12,label='Test')
ax.scatter(prediction[:,0],prediction[:,1],s=12,alpha=.55,label='AE reconstruido')
ax.legend();ax.set_title('Manifold sintético en coordenadas estandarizadas');plt.show()

## Denoising: objetivo limpio a partir de entrada ruidosa
A diferencia de un AE convencional, se construyen pares \((\tilde x, x)\). El ruido no debe contaminar el test al entrenar ni permitir que el modelo vea objetivos futuros. Comparamos frente a la regla ingenua de usar la entrada ruidosa como salida.

In [ ]:
rng=np.random.default_rng(83)
train_clean=ct.clone()
noise_tr=torch.tensor(rng.normal(0,.22,train_clean.shape),dtype=torch.float32)
noisy_train=train_clean+noise_tr
noise_test=torch.tensor(rng.normal(0,.22,cx.shape),dtype=torch.float32)
noisy_test=cx+noise_test
denoise=nn.Sequential(nn.Linear(2,16),nn.Tanh(),nn.Linear(16,1),nn.Tanh(),nn.Linear(1,16),nn.Tanh(),nn.Linear(16,2))
optimizer=torch.optim.Adam(denoise.parameters(),lr=.01)
for epoch in range(250):
    optimizer.zero_grad();loss=((denoise(noisy_train)-train_clean)**2).mean()
    loss.backward();optimizer.step()
with torch.no_grad():denoised=denoise(noisy_test).numpy()
raw_err=np.mean((noisy_test.numpy()-cx.numpy())**2)
clean_err=np.mean((denoised-cx.numpy())**2)
print('Baseline entrada ruidosa:',round(raw_err,5),'Denoising AE:',round(clean_err,5))
print('Resultado exploratorio; falta seleccionar épocas y regularización con validación.')

## Alertas: calibración en validación, métricas en test y costos
La puntuación de reconstrucción no es una probabilidad de fraude. Para practicar una evaluación hay que etiquetar anomalías **artificiales** solo en una copia de test, sin usarlas para ajustar el modelo ni el umbral. La prevalencia de anomalías sintéticas no corresponde a casos de la vida real.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support,average_precision_score
rng=np.random.default_rng(2026)
altered=Xte.copy()
idx=rng.choice(len(altered),size=max(1,len(altered)//10),replace=False)
labels=np.zeros(len(altered),dtype=int);labels[idx]=1
altered[idx]+=rng.choice([-1,1],size=(len(idx),2))*2.5
with torch.no_grad(): altered_hat=model(torch.tensor(altered)).numpy()
scores=np.mean((altered-altered_hat)**2,axis=1)
pred=(scores>cut).astype(int)
prec,rec,f1,_=precision_recall_fscore_support(labels,pred,average='binary',zero_division=0)
print('Umbral calibrado en validación:',round(cut,5),'PR-AUC test sintético:',round(average_precision_score(labels,scores),3))
print('Precisión:',round(prec,3),'Recall:',round(rec,3),'F1:',round(f1,3))
print('Alertas:',pred.sum(),'positivos sintéticos:',labels.sum())

## Protocolo de evaluación y decisión
1. Fijar problema y costo de falsos positivos/negativos.
2. Separar train/validación/test (preferir temporal cuando se anticipa el futuro).
3. Escalar solo con train, ajustar PCA y AE con igual dimensión latente.
4. Seleccionar hiperparámetros y umbral en validación; abrir test una única vez.
5. Repetir semillas, estudiar drift y errores por segmento.
6. Comparar valor económico, no solo MSE; latentes no son factores causales.

**Lecturas primarias:** Baldi & Hornik (1989) https://doi.org/10.1016/0893-6080(89)90014-2 · Kingma & Welling (2013) https://arxiv.org/abs/1312.6114 · Goodfellow et al. (2016), cap. 14 https://www.deeplearningbook.org/.
